In [5]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        os.path.join(dirname, filename)

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [6]:
import pandas as pd
import numpy as np
import time
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, LabelEncoder

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

from xgboost import XGBClassifier

# =====================================================
# LOAD DATA
# =====================================================

df = pd.read_csv(
    "/kaggle/input/datasets/mrwellsdavid/unsw-nb15/UNSW_NB15_training-set.csv"
)

# remove id column
df.drop(columns=['id'], inplace=True, errors='ignore')

# =====================================================
# FEATURES + LABELS
# =====================================================

X = df.drop(columns=['attack_cat', 'label'])

# Stage 1 labels
# 0 = Normal
# 1 = Attack
y_binary = df['label']

# Stage 2 labels
y_attack = df['attack_cat']

# fix label naming
y_attack = y_attack.replace({
    'normal': 'Normal',
    'Normal': 'Normal'
})

# =====================================================
# SPLIT
# =====================================================

X_train, X_val, y_train_bin, y_val_bin, y_train_cat, y_val_cat = train_test_split(
    X,
    y_binary,
    y_attack,
    test_size=0.2,
    stratify=y_binary,
    random_state=42
)

# =====================================================
# FEATURE ENCODING
# =====================================================

cat_cols = [1, 2, 3]

ct = ColumnTransformer([
    ("encoder", OneHotEncoder(handle_unknown="ignore"), cat_cols)
], remainder="passthrough")

X_train = ct.fit_transform(X_train)
X_val = ct.transform(X_val)

# =====================================================
# STAGE 1
# NORMAL vs ATTACK
# =====================================================

print("\n================================================")
print("STAGE 1 : NORMAL vs ATTACK")
print("================================================")

stage1 = RandomForestClassifier(
    n_estimators=500,
    max_depth=30,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features='sqrt',
    criterion='entropy',
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

# =========================
# TRAINING TIME
# =========================

start_train_s1 = time.time()

stage1.fit(X_train, y_train_bin)

end_train_s1 = time.time()

train_time_s1 = end_train_s1 - start_train_s1

# =========================
# TRAIN PREDICTION
# =========================

start_pred_train_s1 = time.time()

train_pred_s1 = stage1.predict(X_train)

end_pred_train_s1 = time.time()

train_pred_time_s1 = end_pred_train_s1 - start_pred_train_s1

# =========================
# VALIDATION PREDICTION
# =========================

start_val_s1 = time.time()

val_pred_s1 = stage1.predict(X_val)

end_val_s1 = time.time()

val_time_s1 = end_val_s1 - start_val_s1

# =====================================================
# TRAIN METRICS - STAGE 1
# =====================================================

print("\nTRAIN RESULTS - STAGE 1\n")

train_acc_s1 = accuracy_score(y_train_bin, train_pred_s1)

train_prec_s1 = precision_score(
    y_train_bin,
    train_pred_s1,
    average='weighted',
    zero_division=0
)

train_rec_s1 = recall_score(
    y_train_bin,
    train_pred_s1,
    average='weighted',
    zero_division=0
)

train_f1_s1 = f1_score(
    y_train_bin,
    train_pred_s1,
    average='weighted',
    zero_division=0
)

print(f"Train Accuracy       : {train_acc_s1:.4f}")
print(f"Train Precision      : {train_prec_s1:.4f}")
print(f"Train Recall         : {train_rec_s1:.4f}")
print(f"Train F1-Score       : {train_f1_s1:.4f}")
print(f"Training Time        : {train_time_s1:.2f} sec")
print(f"Train Prediction Time: {train_pred_time_s1:.2f} sec")

# =====================================================
# VALIDATION METRICS - STAGE 1
# =====================================================

print("\nVALIDATION RESULTS - STAGE 1\n")

val_acc_s1 = accuracy_score(y_val_bin, val_pred_s1)

val_prec_s1 = precision_score(
    y_val_bin,
    val_pred_s1,
    average='weighted',
    zero_division=0
)

val_rec_s1 = recall_score(
    y_val_bin,
    val_pred_s1,
    average='weighted',
    zero_division=0
)

val_f1_s1 = f1_score(
    y_val_bin,
    val_pred_s1,
    average='weighted',
    zero_division=0
)

print(f"Validation Accuracy  : {val_acc_s1:.4f}")
print(f"Validation Precision : {val_prec_s1:.4f}")
print(f"Validation Recall    : {val_rec_s1:.4f}")
print(f"Validation F1-Score  : {val_f1_s1:.4f}")
print(f"Validation Time      : {val_time_s1:.2f} sec")

# =====================================================
# CLASSIFICATION REPORT - STAGE 1
# =====================================================

print("\nCLASSIFICATION REPORT - STAGE 1\n")

print(classification_report(y_val_bin, val_pred_s1))

# =====================================================
# CONFUSION MATRIX - STAGE 1
# =====================================================

cm1 = confusion_matrix(y_val_bin, val_pred_s1)

disp1 = ConfusionMatrixDisplay(
    confusion_matrix=cm1,
    display_labels=["Normal", "Attack"]
)

disp1.plot(
    cmap=plt.cm.Blues
)

plt.title("Stage 1 - Binary Classification")

plt.show()

# =====================================================
# STAGE 2
# ATTACK TYPE CLASSIFICATION
# =====================================================

print("\n================================================")
print("STAGE 2 : ATTACK TYPE CLASSIFICATION")
print("================================================")

# keep attacks only
attack_train_mask = (y_train_bin == 1).to_numpy()

X_train_s2 = X_train[attack_train_mask]

y_train_attack_only = y_train_cat.iloc[attack_train_mask]

# remove Normal
normal_mask = (y_train_attack_only != "Normal").to_numpy()

X_train_s2 = X_train_s2[normal_mask]

y_train_attack_only = y_train_attack_only[normal_mask]

# =====================================================
# LABEL ENCODER
# =====================================================

label_encoder = LabelEncoder()

y_train_s2 = label_encoder.fit_transform(y_train_attack_only)

num_classes = len(np.unique(y_train_s2))

print("\nAttack Classes:\n")

print(label_encoder.classes_)

# =====================================================
# STAGE 2 MODEL
# =====================================================

stage2 = XGBClassifier(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='multi:softmax',
    num_class=num_classes,
    eval_metric='mlogloss',
    n_jobs=-1,
    random_state=42
)

# =====================================================
# TRAINING TIME - STAGE 2
# =====================================================

start_train_s2 = time.time()

stage2.fit(X_train_s2, y_train_s2)

end_train_s2 = time.time()

train_time_s2 = end_train_s2 - start_train_s2

# =====================================================
# TRAIN PREDICTION - STAGE 2
# =====================================================

start_pred_train_s2 = time.time()

train_pred_s2 = stage2.predict(X_train_s2)

end_pred_train_s2 = time.time()

train_pred_time_s2 = end_pred_train_s2 - start_pred_train_s2

# decode predictions
train_pred_s2_labels = label_encoder.inverse_transform(
    train_pred_s2.astype(int)
)

# true labels
y_train_s2_labels = label_encoder.inverse_transform(
    y_train_s2.astype(int)
)

# =====================================================
# VALIDATION DATA - STAGE 2
# =====================================================

attack_val_mask = (y_val_bin == 1).to_numpy()

X_val_s2 = X_val[attack_val_mask]

y_val_attack_only = y_val_cat.iloc[attack_val_mask]

normal_val_mask = (y_val_attack_only != "Normal").to_numpy()

X_val_s2 = X_val_s2[normal_val_mask]

y_val_attack_only = y_val_attack_only[normal_val_mask]

# encode validation labels
y_val_s2 = label_encoder.transform(y_val_attack_only)

# =====================================================
# VALIDATION PREDICTION - STAGE 2
# =====================================================

start_val_s2 = time.time()

val_pred_s2 = stage2.predict(X_val_s2)

end_val_s2 = time.time()

val_time_s2 = end_val_s2 - start_val_s2

# decode labels
val_pred_s2_labels = label_encoder.inverse_transform(
    val_pred_s2.astype(int)
)

y_true_s2_labels = label_encoder.inverse_transform(
    y_val_s2.astype(int)
)

# =====================================================
# TRAIN METRICS - STAGE 2
# =====================================================

print("\nTRAIN RESULTS - STAGE 2\n")

train_acc_s2 = accuracy_score(
    y_train_s2_labels,
    train_pred_s2_labels
)

train_prec_s2 = precision_score(
    y_train_s2_labels,
    train_pred_s2_labels,
    average='weighted',
    zero_division=0
)

train_rec_s2 = recall_score(
    y_train_s2_labels,
    train_pred_s2_labels,
    average='weighted',
    zero_division=0
)

train_f1_s2 = f1_score(
    y_train_s2_labels,
    train_pred_s2_labels,
    average='weighted',
    zero_division=0
)

print(f"Train Accuracy       : {train_acc_s2:.4f}")
print(f"Train Precision      : {train_prec_s2:.4f}")
print(f"Train Recall         : {train_rec_s2:.4f}")
print(f"Train F1-Score       : {train_f1_s2:.4f}")
print(f"Training Time        : {train_time_s2:.2f} sec")
print(f"Train Prediction Time: {train_pred_time_s2:.2f} sec")

# =====================================================
# VALIDATION METRICS - STAGE 2
# =====================================================

print("\nVALIDATION RESULTS - STAGE 2\n")

val_acc_s2 = accuracy_score(
    y_true_s2_labels,
    val_pred_s2_labels
)

val_prec_s2 = precision_score(
    y_true_s2_labels,
    val_pred_s2_labels,
    average='weighted',
    zero_division=0
)

val_rec_s2 = recall_score(
    y_true_s2_labels,
    val_pred_s2_labels,
    average='weighted',
    zero_division=0
)

val_f1_s2 = f1_score(
    y_true_s2_labels,
    val_pred_s2_labels,
    average='weighted',
    zero_division=0
)

print(f"Validation Accuracy  : {val_acc_s2:.4f}")
print(f"Validation Precision : {val_prec_s2:.4f}")
print(f"Validation Recall    : {val_rec_s2:.4f}")
print(f"Validation F1-Score  : {val_f1_s2:.4f}")
print(f"Validation Time      : {val_time_s2:.2f} sec")

# =====================================================
# CLASSIFICATION REPORT - STAGE 2
# =====================================================

print("\nCLASSIFICATION REPORT - STAGE 2\n")

print(classification_report(
    y_true_s2_labels,
    val_pred_s2_labels
))

# =====================================================
# CONFUSION MATRIX - STAGE 2
# =====================================================

cm2 = confusion_matrix(
    y_true_s2_labels,
    val_pred_s2_labels
)

disp2 = ConfusionMatrixDisplay(
    confusion_matrix=cm2,
    display_labels=label_encoder.classes_
)

disp2.plot(
    cmap=plt.cm.Blues,
    xticks_rotation=45
)

plt.title("Stage 2 - Attack Classification")

plt.show()

# =====================================================
# FINAL HIERARCHICAL IDS
# =====================================================

print("\n================================================")
print("FINAL HIERARCHICAL IDS")
print("================================================")

# =========================
# TRAIN PREDICTION
# =========================

stage1_train_pred = stage1.predict(X_train)

final_train_pred = np.array(
    ["Normal"] * X_train.shape[0],
    dtype=object
)

attack_train_idx = (stage1_train_pred == 1)

if attack_train_idx.sum() > 0:

    stage2_train_all = stage2.predict(
        X_train[attack_train_idx]
    )

    stage2_train_all_labels = label_encoder.inverse_transform(
        stage2_train_all.astype(int)
    )

    final_train_pred[attack_train_idx] = stage2_train_all_labels

# =========================
# VALIDATION PREDICTION
# =========================

final_val_pred = np.array(
    ["Normal"] * X_val.shape[0],
    dtype=object
)

attack_val_idx = (val_pred_s1 == 1)

if attack_val_idx.sum() > 0:

    stage2_val_all = stage2.predict(
        X_val[attack_val_idx]
    )

    stage2_val_all_labels = label_encoder.inverse_transform(
        stage2_val_all.astype(int)
    )

    final_val_pred[attack_val_idx] = stage2_val_all_labels

# =====================================================
# TRAIN METRICS - FINAL IDS
# =====================================================

print("\nTRAIN RESULTS - FINAL IDS\n")

final_train_acc = accuracy_score(
    y_train_cat,
    final_train_pred
)

final_train_prec = precision_score(
    y_train_cat,
    final_train_pred,
    average='weighted',
    zero_division=0
)

final_train_rec = recall_score(
    y_train_cat,
    final_train_pred,
    average='weighted',
    zero_division=0
)

final_train_f1 = f1_score(
    y_train_cat,
    final_train_pred,
    average='weighted',
    zero_division=0
)

print(f"Train Accuracy  : {final_train_acc:.4f}")
print(f"Train Precision : {final_train_prec:.4f}")
print(f"Train Recall    : {final_train_rec:.4f}")
print(f"Train F1-Score  : {final_train_f1:.4f}")

# =====================================================
# VALIDATION METRICS - FINAL IDS
# =====================================================

print("\nVALIDATION RESULTS - FINAL IDS\n")

final_val_acc = accuracy_score(
    y_val_cat,
    final_val_pred
)

final_val_prec = precision_score(
    y_val_cat,
    final_val_pred,
    average='weighted',
    zero_division=0
)

final_val_rec = recall_score(
    y_val_cat,
    final_val_pred,
    average='weighted',
    zero_division=0
)

final_val_f1 = f1_score(
    y_val_cat,
    final_val_pred,
    average='weighted',
    zero_division=0
)

print(f"Validation Accuracy  : {final_val_acc:.4f}")
print(f"Validation Precision : {final_val_prec:.4f}")
print(f"Validation Recall    : {final_val_rec:.4f}")
print(f"Validation F1-Score  : {final_val_f1:.4f}")

# =====================================================
# CLASSIFICATION REPORT - FINAL IDS
# =====================================================

print("\nCLASSIFICATION REPORT - FINAL IDS\n")

print(classification_report(
    y_val_cat,
    final_val_pred
))

# =====================================================
# CONFUSION MATRIX - FINAL IDS
# =====================================================

all_labels = np.unique(
    np.concatenate([
        y_val_cat.unique(),
        final_val_pred
    ])
)

cm_final = confusion_matrix(
    y_val_cat,
    final_val_pred,
    labels=all_labels
)

disp_final = ConfusionMatrixDisplay(
    confusion_matrix=cm_final,
    display_labels=all_labels
)

disp_final.plot(
    cmap=plt.cm.Blues,
    xticks_rotation=45
)

plt.title("Final Hierarchical IDS")

plt.show()

print("\nDONE SUCCESSFULLY")


STAGE 1 : NORMAL vs ATTACK


NameError: name 'RandomForestClassifier' is not defined